In [9]:
from sklearn.datasets import make_classification
import pandas as pd

X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_classes=2,
    weights=[0.9, 0.1],  # 90% placed, 10% unplaced
    random_state=42
)

df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(20)])
df['target'] = y

In [10]:
from sklearn.model_selection import train_test_split

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data
X_test_scaled = scaler.transform(X_test)

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_scaled, y_train)

y_pred = rf.predict(X_test_scaled)

baseline_acc = accuracy_score(y_test, y_pred)
baseline_f1 = f1_score(y_test, y_pred)

print("Baseline Accuracy:", baseline_acc)
print("Baseline F1 Score:", baseline_f1)

Baseline Accuracy: 0.91
Baseline F1 Score: 0.3076923076923077


In [13]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

grid_acc = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)

grid_acc.fit(X_train_scaled, y_train)

print("Best Params (Accuracy):", grid_acc.best_params_)

Best Params (Accuracy): {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 50}


In [14]:
grid_f1 = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1
)

grid_f1.fit(X_train_scaled, y_train)

print("Best Params (F1):", grid_f1.best_params_)

Best Params (F1): {'max_depth': None, 'min_samples_split': 10, 'n_estimators': 50}


In [15]:
import time

start = time.time()

grid_f1.fit(X_train_scaled, y_train)

grid_time = time.time() - start
print("Grid Search Time:", grid_time)

Grid Search Time: 10.238751649856567


In [16]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

param_dist = {
    'n_estimators': np.arange(10, 500),
    'max_depth': [None] + list(np.arange(5, 50)),
    'min_samples_split': np.arange(2, 20)
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    scoring='f1',
    cv=5,
    random_state=42,
    n_jobs=-1
)

start = time.time()

random_search.fit(X_train_scaled, y_train)

random_time = time.time() - start

print("Best Params (Random):", random_search.best_params_)

Best Params (Random): {'n_estimators': np.int64(398), 'min_samples_split': np.int64(2), 'max_depth': np.int64(48)}


In [17]:
results = pd.DataFrame({
    "Method": ["Grid Search", "Random Search"],
    "Time (seconds)": [grid_time, random_time],
    "Best F1 Score": [grid_f1.best_score_, random_search.best_score_]
})

print(results)

          Method  Time (seconds)  Best F1 Score
0    Grid Search       10.238752       0.403463
1  Random Search       16.700154       0.397733
